# Modelado Basico Bank Marketing

Notebook para entrenar un modelo base usando el flujo ya preprocesado. La validacion final sigue reservada para `Test`.

## Objetivo

Tomar el pipeline de preprocesamiento, entrenar un clasificador basal y comparar su desempeno con una linea base simple.

## Regla metodologica

1. Reutilizar el split y el preprocessing definidos en el notebook anterior.
2. Ajustar el pipeline solo con `Train`.
3. Evaluar primero un baseline simple.
4. Comparar con un clasificador supervisado.
5. Reportar metricas sobre `Test` sin tocarlo durante el ajuste.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import importlib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import preprocessing as preprocessing_module
importlib.reload(preprocessing_module)
from preprocessing import load_data, split_data, build_preprocessing_pipeline

DATA_PATH = '../data/raw/bank-additional-full.csv'
TARGET = 'y'

df = load_data(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(df, TARGET)

numeric_cols = ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
nominal_cols = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
ordinal_cols = ['education']

preprocessor = build_preprocessing_pipeline(numeric_cols, nominal_cols, ordinal_cols)
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print('Shapes:', X_train_p.shape, X_test_p.shape)


Shapes: (32950, 50) (8238, 50)


In [2]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, positive_label='yes'):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)
        pos_index = list(model.classes_).index(positive_label)
        score = proba[:, pos_index]
        roc = roc_auc_score((y_te == positive_label).astype(int), score)
    else:
        roc = np.nan
    return {
        'accuracy': accuracy_score(y_te, pred),
        'precision': precision_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'recall': recall_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'f1': f1_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'roc_auc': roc,
        'confusion_matrix': confusion_matrix(y_te, pred),
        'report': classification_report(y_te, pred, zero_division=0),
    }

baseline = DummyClassifier(strategy='most_frequent')
baseline_result = evaluate_model(baseline, X_train_p, y_train, X_test_p, y_test)
display(pd.DataFrame([baseline_result]).drop(columns=['confusion_matrix', 'report']))
print(baseline_result['confusion_matrix'])
print(baseline_result['report'])


,accuracy,precision,recall,f1,roc_auc
0,0.887351,0.0,0.0,0.0,0.5


[[7310    0]
 [ 928    0]]
              precision    recall  f1-score   support

          no       0.89      1.00      0.94      7310
         yes       0.00      0.00      0.00       928

    accuracy                           0.89      8238
   macro avg       0.44      0.50      0.47      8238
weighted avg       0.79      0.89      0.83      8238



In [3]:
logit = LogisticRegression(max_iter=1000, class_weight='balanced')
logit_result = evaluate_model(logit, X_train_p, y_train, X_test_p, y_test)
display(pd.DataFrame([logit_result]).drop(columns=['confusion_matrix', 'report']))
print(logit_result['confusion_matrix'])
print(logit_result['report'])


,accuracy,precision,recall,f1,roc_auc
0,0.832969,0.363747,0.644397,0.465008,0.801189


[[6264 1046]
 [ 330  598]]
              precision    recall  f1-score   support

          no       0.95      0.86      0.90      7310
         yes       0.36      0.64      0.47       928

    accuracy                           0.83      8238
   macro avg       0.66      0.75      0.68      8238
weighted avg       0.88      0.83      0.85      8238



In [4]:
model_specs = {
    'baseline_majority': DummyClassifier(strategy='most_frequent'),
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'decision_tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
}

results = []
for name, model in model_specs.items():
    result = evaluate_model(model, X_train_p, y_train, X_test_p, y_test)
    result['model'] = name
    results.append(result)

results_df = pd.DataFrame(results).drop(columns=['confusion_matrix', 'report'])
results_df = results_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].sort_values('f1', ascending=False)
display(results_df)


,model,accuracy,precision,recall,f1,roc_auc
3,random_forest,0.861617,0.423851,0.635776,0.508621,0.813440
2,decision_tree,0.833576,0.366163,0.653017,0.469222,0.792639
1,logistic_regression,0.832969,0.363747,0.644397,0.465008,0.801189
0,baseline_majority,0.887351,0.000000,0.000000,0.000000,0.500000


In [5]:
best_model_name = results_df.iloc[0]['model']
best_model_name


'random_forest'

## Lectura del modelo

El baseline sirve como piso metodologico. LogisticRegression actua como primer clasificador interpretable para comparar el efecto del preprocesamiento y de las features nuevas.

## Siguientes pasos

1. Probar uno o dos modelos adicionales.
2. Ajustar umbrales y revisar el balance entre precision y recall.
3. Elegir el modelo final para la defensa.